# 11. Face Detection - SCRFD500M

| 항목 | 내용 |
|---|---|
| 대상 모델 | `SCRFD500M.dxnn` |
| 카테고리 | **Face Detection** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 비율을 유지하며 여백을 넣는 전처리가 좌표 복원에 남기는 값을 계산해 제시할 수 있다.
2. 아홉 개 출력을 스케일별 세 묶음으로 묶는 판정 근거를 채널 수와 개수로 설명할 수 있다.
3. 격자 자리와 예측값에서 사각형 좌표를 만드는 계산을 실제 숫자로 수행할 수 있다.
4. 후보 개수가 임계값과 NMS 를 거쳐 줄어드는 과정을 단계별 개수로 제시할 수 있다.

## 1. Face Detection (얼굴 검출)

**영상에서 얼굴이 있는 자리를 모두 찾아 각각의 위치와 크기를 사각형으로 답하는 작업**

개수가 정해져 있지 않다는 점이 분류·속성 계열과 다르다. 얼굴이 없으면 답이 0개이고, 여럿이면 그만큼 나온다.
검출은 뒤에 붙는 작업의 앞단이 되는 일이 많아 **잘라 낼 영역을 만들어 주는 역할**을 한다.

**출력의 성격**

좌표와 점수의 목록이다. 같은 얼굴을 여러 후보가 동시에 가리키므로 겹친 후보를 정리하는 단계가 반드시 필요하고,
그 정리를 어떻게 하느냐가 결과 개수를 바꾼다.

**활용 사례**

- 얼굴 인식·속성 판정의 앞단으로 얼굴 영역 잘라내기
- 촬영 기기의 얼굴 초점·노출 보정
- 화면 속 얼굴 가리기

**포함되지 않는 작업**

- 누구인지 판정 (Face Embedding)
- 표정·나이 판정 (Face Attribute)
- 얼굴 윤곽을 픽셀 단위로 나누기 (Instance Segmentation)

## 2. 전체 파이프라인

```text
sample_face.jpg  1536×1024 BGR
  → 전처리         BGR→RGB · 비율 유지 축소 640×427 · 위아래 여백 106·107 → 640×640
  → run()          입력 Tensor 하나를 전달
  → npu_0          스케일별 중간 tensor 9개 (내부 전달, Public 아님)
  → cpu_0          score_8/16/32 · bbox_8/16/32 · kps_8/16/32  (Public Output 9개)
  → 후처리         스케일별로 묶기 → 격자 좌표로 decode → 임계값 → NMS → 여백 제거·원본 좌표
  → Result         FaceResult (box · confidence · 5점 keypoint)
  → 시각화         사각형·점수·5점을 원본 위에 그림
```

![11. Face Detection 전체 파이프라인](resources/11_Face_Detection_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `input.1` |
| Shape / Dtype | `[1, 640, 640, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | **Letterbox — 비율 유지 후 여백 채우기** |
| Normalization | 없음 |

**비율을 유지하는 것이 이 계열의 특징이다.** 앞의 모델들은 사진을 눌러 넣었지만, 검출은 **좌표를 답하므로**
눌러 넣으면 사각형이 함께 찌그러진다. 그래서 긴 쪽을 기준으로 줄이고 남는 자리를 회색(114, 114, 114)으로 채운다.

| 값 | 이번 실행 | 뜻 |
|---|---:|---|
| `gain` | 0.416667 | 축소 비율. `min(640/1024, 640/1536)` |
| 줄인 크기 | 640×427 | `1536 × 0.4167 = 640`, `1024 × 0.4167 = 427` |
| `pad_y` | 106 | 위쪽 여백. 아래쪽은 107 |
| `pad_x` | 0 | 가로는 꽉 차서 여백이 없음 |

이 세 값(`gain`·`pad_x`·`pad_y`)을 Context 에 담아 두어야 후처리가 좌표를 원본으로 되돌릴 수 있다. **여백을
빼지 않고 나누기만 하면 결과가 위로 밀린다.**

### 3.2 출력

이 모델은 Task 가 둘이며 **후보를 정리해 내놓는 일이 CPU Task 의 몫**이다. `npu_0` 이 낸 9개 중간 tensor 를
`cpu_0` 이 받아 후보 목록 형태로 바꾼다.

| 생성 Task | Tensor | shape / dtype | 다음 사용자 | Public |
|---|---|---|---|---|
| `npu_0` | `429`~`517` 9개 | `[1, C, H, W]` / `float32` | `cpu_0` | 아님 — 내부 전달 |
| `cpu_0` | `score_8` · `bbox_8` · `kps_8` | `[1, 12800, 1·4·10]` / `float32` | Host 후처리 | CPU Task 출력 (= Public 출력) |
| `cpu_0` | `score_16` · `bbox_16` · `kps_16` | `[1, 3200, 1·4·10]` / `float32` | Host 후처리 | CPU Task 출력 (= Public 출력) |
| `cpu_0` | `score_32` · `bbox_32` · `kps_32` | `[1, 800, 1·4·10]` / `float32` | Host 후처리 | CPU Task 출력 (= Public 출력) |

**왜 세 벌인가 — 얼굴 크기가 제각각이기 때문이다.** 작은 얼굴은 촘촘한 격자로 찾아야 하고 큰 얼굴은 넓은 격자로
찾는 편이 안정적이다. 그래서 세 해상도의 격자를 동시에 쓰고, 격자 한 칸의 간격을 `stride` 라 부른다.

| stride | 격자 | 자리당 후보 | 후보 수 | 주로 잡는 얼굴 |
|---:|---|---:|---:|---|
| 8 | 80×80 | 2 | 12,800 | 작은 얼굴 |
| 16 | 40×40 | 2 | 3,200 | 중간 얼굴 |
| 32 | 20×20 | 2 | 800 | 큰 얼굴 |

`80 × 80 × 2 = 12,800` 이며 세 벌을 더하면 후보가 **16,800개**다. 이 중 얼굴이 있는 자리만 골라내는 것이
후처리의 일이다.

**아홉 개를 이름이 아니라 모양으로 묶는다.** 이름이 친절하게 붙어 있지만 후처리는 이름을 읽지 않고 **후보 수와
채널 수**로 판정한다. 컴파일러가 이름을 바꾸어도 동작하도록 만든 구조다.

| 채널 수 | 뜻 |
|---:|---|
| 1 | 그 자리에 얼굴이 있을 점수 |
| 4 | 격자 중심에서 사각형 네 변까지의 거리 |
| 10 | 5점 keypoint 의 (x, y) 오프셋 |

같은 후보 수를 가진 셋이 한 스케일의 묶음이 되고, `stride` 는 후보 수에서 거꾸로 계산한다 —
`640 / sqrt(12800 / 2) = 8` 이다. 2 로 나누는 것은 **격자 한 자리마다 후보가 두 개씩** 붙기 때문이다.
`80×80×2 = 12800`, `40×40×2 = 3200`, `20×20×2 = 800` 으로 세 스케일이 모두 맞는다.

**사각형을 만드는 계산**

이 모델은 앵커 박스를 쓰지 않고 **격자 중심에서 각 변까지의 거리**를 예측한다. 거리는 `stride` 단위로 나오므로
곱해서 화소 단위로 바꾼다.

```text
cx = gx × stride,  cy = gy × stride
x1 = cx - d_left  × stride      x2 = cx + d_right  × stride
y1 = cy - d_top   × stride      y2 = cy + d_bottom × stride
```

keypoint 도 같은 방식이며 중심에서의 오프셋을 더한다. `kx = cx + offset_x × stride` 다.

**5점 keypoint 의 번호와 부위**

| index | 부위 | 색 (BGR) |
|---:|---|---|
| 0 | left_eye | (0, 255, 0) |
| 1 | right_eye | (0, 255, 0) |
| 2 | nose | (255, 128, 0) |
| 3 | left_mouth | (255, 0, 128) |
| 4 | right_mouth | (255, 0, 128) |

전체 설명: [SCRFD (InsightFace)](https://insightface.ai/scrfd)

**`left`·`right` 는 화면 기준이다.** 이 보드에서 실행해 좌표를 확인한 결과 0번이 화면 왼쪽 눈에 놓였다. 인물
기준으로 적는 모델도 있으므로 다른 모델의 기준을 그대로 옮기지 않는다.

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. `cpu_0` 이 9개
Public Output 을 내놓고 후보 수가 12800·3200·800 인지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "SCRFD500M.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

### 3.3 Config.json

| key | 현재 값 | 기본값 | 의미 |
|---|---:|---:|---|
| `score_threshold` | 0.3 | 0.3 | 이 점수에 못 미치는 후보를 버림 |
| `nms_threshold` | 0.4 | 0.4 | 겹침이 이 비율을 넘으면 낮은 점수를 버림 |

두 값이 코드 기본값과 같다. 점수 임계값을 낮추면 흐릿한 얼굴도 남고 잘못된 후보도 늘며, 겹침 임계값을 낮추면
나란히 붙은 얼굴 하나가 지워질 수 있다.

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정하고 설정 파일을 읽는다. 이후 셀은 여기서 만든 이름을 그대로 쓴다.

In [ ]:
import json
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

sys.dont_write_bytecode = True
IMAGE_PATH = DX_APP / "sample" / "img/sample_face.jpg"
CONFIG_PATH = DX_APP / "src" / "python_example" / "face_detection" / "scrfd500m" / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.is_file() else {}

FACE_KEYPOINT_NAMES = ["left_eye", "right_eye", "nose", "left_mouth", "right_mouth"]
FACE_KEYPOINT_COLORS = [(0, 255, 0), (0, 255, 0), (255, 128, 0), (255, 0, 128), (255, 0, 128)]

print("config", CONFIG)

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 이 모델은 `scale`(gain)·`pad_x`·
  `pad_y` 가 좌표 복원의 근거다
- `Keypoint` · `FaceResult` — 후처리가 만들어 시각화로 넘기는 결과. 사각형·점수·5점을 담는다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale: float = 1.0
    pad_x: int = 0
    pad_y: int = 0


@dataclass
class Keypoint:
    """점 하나의 좌표와 신뢰도. 원본 대응: common/base/i_processor.py"""

    x: float = 0.0
    y: float = 0.0
    confidence: float = 0.0


@dataclass
class FaceResult:
    """얼굴 하나의 사각형·점수·5점. 원본 대응: common/processors/face_postprocessor.py"""

    box: list = field(default_factory=list)
    confidence: float = 0.0
    class_id: int = 0
    keypoints: list = field(default_factory=list)


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. 출력 9개의 이름과 후보 수를 함께 확인한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output (런타임 순서):")
for index, info in enumerate(output_info):
    print(f"  [{index}] {info['name']:10s} {info['shape']}")

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

비율을 유지해 줄인 뒤 남는 자리를 회색으로 채워 640×640 을 만든다. 좌표를 답하는 작업이므로 **눌러 넣지
않는 것이 핵심**이며, 되돌리는 데 필요한 `gain`·`pad_x`·`pad_y` 를 Context 에 적어 둔다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 640×640, 그중 실제 사진은 640×427
- 여백 — 위 106 · 아래 107 · 좌우 0
- Context — `scale`(gain) 과 `pad_x`·`pad_y`

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다. 얼굴이 여러 개 있는 장면이라 검출 결과의 개수를 눈으로 셀 수 있다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}", figsize=(11, 7))

#### 전처리 함수 정의

- `scrfd_preprocessor_create` — 모델 입력 크기와 여백 색을 state 에 담음
- `scrfd_letterbox` — 비율 유지 축소와 여백 채우기
- `scrfd_preprocessor_process` — letterbox 를 적용하고 복원용 값을 Context 에 담음

In [ ]:
def scrfd_preprocessor_create(input_width: int, input_height: int,
                              pad_color: tuple = (114, 114, 114)) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height, "pad_color": pad_color}


def scrfd_letterbox(state: dict, image: np.ndarray) -> tuple:
    """비율을 유지해 줄이고 남는 자리를 채운다. 원본 대응: LetterboxPreprocessor._letterbox()"""
    height, width = image.shape[:2]
    gain = min(state["input_height"] / height, state["input_width"] / width)
    new_size = int(round(width * gain)), int(round(height * gain))

    delta_w = (state["input_width"] - new_size[0]) / 2
    delta_h = (state["input_height"] - new_size[1]) / 2
    if (width, height) != new_size:
        image = cv2.resize(image, new_size, interpolation=cv2.INTER_LINEAR)

    top, bottom = int(round(delta_h - 0.1)), int(round(delta_h + 0.1))
    left, right = int(round(delta_w - 0.1)), int(round(delta_w + 0.1))
    padded = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT,
                                value=state["pad_color"])
    return padded, gain, (top, left)


def scrfd_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    padded, gain, pad = scrfd_letterbox(state, rgb)
    ctx.scale = gain
    ctx.pad_y, ctx.pad_x = pad
    return padded, ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만들고 여백이 들어간 모양을 그림으로 확인한다. 위아래 회색 띠가 좌표를 되돌릴 때
빼야 하는 값이다.

In [ ]:
preprocessor_state = scrfd_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = scrfd_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype} "
      f"contiguous={input_tensor.flags['C_CONTIGUOUS']}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"gain={ctx.scale:.6f} pad_x={ctx.pad_x} pad_y={ctx.pad_y}")
print(f"실제 사진 영역  {int(round(ctx.original_width * ctx.scale))}x"
      f"{int(round(ctx.original_height * ctx.scale))} / 나머지는 여백")

notebook_show_bgr(cv2.cvtColor(input_tensor, cv2.COLOR_RGB2BGR),
                  f"Letterboxed {INPUT_WIDTH}x{INPUT_HEIGHT} (gray bands are padding)",
                  figsize=(6, 6))

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 넘기고 Public Output 9개를 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복 비용이
포함돼 있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(f"  [{index}] {output_info[index]['name']:10s} {str(array.shape):16s} "
          f"min={array.min():8.4f} max={array.max():8.4f}")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- 후보 수 — 12800 · 3200 · 800 세 종류
- 채널 수 — 1(점수) · 4(거리) · 10(오프셋)
- 점수의 최댓값 — 어느 스케일에서 얼굴이 잡혔는지
- 임계값 통과 개수 — 스케일마다 몇 개인지

In [ ]:
score_threshold = CONFIG.get("score_threshold", 0.3)
buckets = {}
for index, output in enumerate(outputs):
    array = np.asarray(output)
    buckets.setdefault(array.shape[1], {})[array.shape[2]] = array

print(f"후보 수 종류: {sorted(buckets, reverse=True)}")
total_candidates = 0
for count in sorted(buckets, reverse=True):
    stride = INPUT_WIDTH // int(round(np.sqrt(count // 2)))
    scores = buckets[count][1].reshape(-1)
    total_candidates += scores.size
    print(f"  n={count:6d} stride={stride:2d} 채널 {sorted(buckets[count])} "
          f"점수 최대={scores.max():.4f} {score_threshold} 초과={int((scores > score_threshold).sum())}개")
print(f"후보 총 {total_candidates:,}개")

### 4.4 후처리

아홉 tensor 를 스케일별로 묶고, 격자 자리와 예측 거리에서 사각형을 만들고, 점수가 낮은 후보를 버린 뒤 겹친
후보를 정리한다. 마지막으로 여백을 빼고 `gain` 으로 나누어 원본 좌표로 되돌린다.

**순서를 바꿀 수 없다.** 겹침 정리는 사각형이 만들어진 뒤에만 할 수 있고, 좌표 복원은 정리가 끝난 뒤에 하는
편이 계산이 적다. 좌표 복원을 먼저 하면 남길 후보만 골라도 될 계산을 16,800번 하게 된다.

**겹침 정리는 점수 임계값과 함께 한 함수에서 이루어진다.** `cv2.dnn.NMSBoxes` 가 점수 임계값과 겹침 임계값을
함께 받아, 낮은 점수를 버리고 겹친 것 중 가장 높은 점수만 남긴다. 그래서 코드에는 임계값으로 걸러내는 별도
단계가 없다.

**좌표 복원 계산**

```text
원본 x = (입력 x - pad_x) / gain      원본 y = (입력 y - pad_y) / gain
```

이번 실행의 1위 후보는 입력 좌표에서 `(280.9, 169.0)` 이고 `pad_y = 106`, `gain = 0.4167` 이므로 원본에서
`(280.9 - 0) / 0.4167 = 674.1`, `(169.0 - 106) / 0.4167 = 151.2` 가 된다. 마지막으로 사진 밖으로 나간 값은
사진 경계로 잘라 낸다.

#### 후처리 함수 정의 — 묶기와 decode

- `scrfd_postprocessor_create` — 임계값·점 개수·자리당 후보 수를 state 에 담음
- `scrfd_group_by_scale` — 후보 수와 채널 수로 아홉 tensor 를 세 묶음으로 정리
- `scrfd_decode_scale` — 격자 자리와 예측 거리에서 사각형·점 좌표를 만듦

In [ ]:
def scrfd_postprocessor_create(input_width: int, config: dict) -> dict:
    """Config 와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {
        "input_width": input_width,
        "score_threshold": config.get("score_threshold", 0.3),
        "nms_threshold": config.get("nms_threshold", 0.4),
        "num_keypoints": config.get("num_keypoints", 5),
        "num_anchors": config.get("num_anchors", 2),
    }


def scrfd_group_by_scale(state: dict, outputs: list) -> list:
    """후보 수와 채널 수로 (stride, score, bbox, kps) 묶음을 만든다.

    원본 대응: SCRFDPostprocessor._group_tensors_by_spatial() · _build_triplets()
    """
    buckets = {}
    for output in outputs:
        array = np.asarray(output)
        if array.ndim != 3 or array.shape[0] != 1:
            continue
        count, channels = array.shape[1], array.shape[2]
        entry = buckets.setdefault(count, {})
        if channels == 1:
            entry["score"] = array
        elif channels == 4:
            entry["bbox"] = array
        elif channels == state["num_keypoints"] * 2:
            entry["kps"] = array

    triplets = []
    for count, entry in buckets.items():
        if {"score", "bbox", "kps"} <= entry.keys():
            stride = state["input_width"] // max(1, int(round(np.sqrt(count // state["num_anchors"]))))
            triplets.append((stride, entry["score"], entry["bbox"], entry["kps"]))
    return triplets


def scrfd_decode_scale(state: dict, stride: int, score_t: np.ndarray, bbox_t: np.ndarray,
                       kps_t: np.ndarray) -> tuple:
    """격자 자리와 예측 거리에서 사각형·점 좌표를 만든다.

    원본 대응: SCRFDPostprocessor._decode_all_triplets()
    """
    score = score_t.reshape(-1)
    bbox = bbox_t.reshape(-1, 4)
    kps = kps_t.reshape(-1, state["num_keypoints"] * 2)

    count = score.size
    grid = max(1, int(round(np.sqrt(count // state["num_anchors"]))))
    location = np.arange(count) // state["num_anchors"]
    center_x = (location % grid) * stride
    center_y = (location // grid) * stride

    x1 = center_x - bbox[:, 0] * stride
    y1 = center_y - bbox[:, 1] * stride
    x2 = center_x + bbox[:, 2] * stride
    y2 = center_y + bbox[:, 3] * stride

    keypoint_x = center_x[:, None] + kps[:, 0::2] * stride
    keypoint_y = center_y[:, None] + kps[:, 1::2] * stride

    return (np.column_stack([x1, y1, x2 - x1, y2 - y1]), score,
            np.stack((keypoint_x, keypoint_y), axis=-1).reshape(count, -1))

#### 후처리 함수 정의 — NMS 와 좌표 복원

- `scrfd_postprocessor_process` — 세 스케일을 이어 붙여 NMS 를 걸고, 남은 후보를 원본 좌표로 되돌려 Result 생성

In [ ]:
def scrfd_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    boxes_list, scores_list, keypoints_list = [], [], []
    for stride, score_t, bbox_t, kps_t in scrfd_group_by_scale(state, outputs):
        boxes, scores, keypoints = scrfd_decode_scale(state, stride, score_t, bbox_t, kps_t)
        boxes_list.append(boxes)
        scores_list.append(scores)
        keypoints_list.append(keypoints)
    if not boxes_list:
        return []

    boxes = np.vstack(boxes_list)
    scores = np.concatenate(scores_list)
    keypoints = np.vstack(keypoints_list)

    indices = cv2.dnn.NMSBoxes(boxes.tolist(), scores.tolist(),
                               state["score_threshold"], state["nms_threshold"])
    if len(indices) == 0:
        return []

    gain = max(ctx.scale, 1e-6)
    results = []
    for index in np.array(indices).reshape(-1):
        x1, y1, width, height = boxes[index]
        box = [
            float(np.clip((x1 - ctx.pad_x) / gain, 0, ctx.original_width - 1)),
            float(np.clip((y1 - ctx.pad_y) / gain, 0, ctx.original_height - 1)),
            float(np.clip((x1 + width - ctx.pad_x) / gain, 0, ctx.original_width - 1)),
            float(np.clip((y1 + height - ctx.pad_y) / gain, 0, ctx.original_height - 1)),
        ]
        points = [
            Keypoint(x=float(np.clip((point[0] - ctx.pad_x) / gain, 0, ctx.original_width - 1)),
                     y=float(np.clip((point[1] - ctx.pad_y) / gain, 0, ctx.original_height - 1)),
                     confidence=1.0)
            for point in keypoints[index].reshape(-1, 2)
        ]
        results.append(FaceResult(box=box, confidence=float(scores[index]), class_id=0,
                                  keypoints=points))
    return results

#### 후처리 실행

확인할 것:

- 단계별 개수 — 후보 16,800개에서 몇 개가 남는지
- 각 결과의 점수와 사각형 — 원본 크기 안에 들어오는지
- keypoint 개수 — 얼굴마다 5개인지
- 임계값을 바꿨을 때 — 결과 개수가 어떻게 달라지는지

In [ ]:
postprocessor_state = scrfd_postprocessor_create(INPUT_WIDTH, CONFIG)
results = scrfd_postprocessor_process(postprocessor_state, outputs, ctx)

print(f"후보 {total_candidates:,}개 → 점수 {score_threshold} 초과와 겹침 정리를 거쳐 {len(results)}개")
for order, face in enumerate(results, start=1):
    x1, y1, x2, y2 = face.box
    print(f"  #{order} score={face.confidence:.4f} box=({x1:7.1f}, {y1:6.1f}, {x2:7.1f}, {y2:6.1f}) "
          f"keypoints={len(face.keypoints)}개")

print("\n1위 얼굴의 5점:")
for index, point in enumerate(results[0].keypoints):
    print(f"  {index} {FACE_KEYPOINT_NAMES[index]:12s} ({point.x:7.1f}, {point.y:6.1f})")

for threshold in (0.2, 0.3, 0.5, 0.8, 0.95):
    state = scrfd_postprocessor_create(INPUT_WIDTH, {**CONFIG, "score_threshold": threshold})
    print(f"score_threshold={threshold} → {len(scrfd_postprocessor_process(state, outputs, ctx))}개")

### 4.5 시각화

#### Visualizer 코드

결과 객체만 받아 사각형·점수 라벨·5점을 원본 위에 그린다. 점은 부위별로 색을 달리하고, 라벨은 사각형 위에
채운 상자를 깔아 글자가 배경에 묻히지 않게 한다. 그림에 넣는 글자는 영문만 쓴다 — OpenCV 기본 폰트에 한글
자모가 없어 한글을 넣으면 글자가 깨진다.

In [ ]:
def scrfd_visualizer_visualize(image: np.ndarray, results: list, keypoint_radius: int = 3,
                               box_color: tuple = (0, 255, 0)) -> np.ndarray:
    """Result 를 원본 이미지 위에 그린다. 원본 대응: Runner.visualize()"""
    output = image.copy()

    for face in results:
        x1, y1, x2, y2 = [int(value) for value in face.box]
        cv2.rectangle(output, (x1, y1), (x2, y2), box_color, 2)

        label = f"face: {face.confidence:.2f}"
        (label_w, label_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        label_y = y1 - 10 if y1 - 10 > label_h else y1 + 10
        cv2.rectangle(output, (x1, label_y - label_h), (x1 + label_w, label_y + label_h),
                      box_color, cv2.FILLED)
        cv2.putText(output, label, (x1, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1,
                    cv2.LINE_AA)

        for index, point in enumerate(face.keypoints):
            color = FACE_KEYPOINT_COLORS[index] if index < len(FACE_KEYPOINT_COLORS) else (0, 255, 255)
            cv2.circle(output, (int(round(point.x)), int(round(point.y))), keypoint_radius,
                       color, -1)

    return output

#### 결과 시각화

검출 결과를 inline 으로 표시한다. 사각형이 얼굴에 맞고 5점이 눈·코·입에 놓이는지 확인한다.

In [ ]:
output_image = scrfd_visualizer_visualize(image, results)
notebook_show_bgr(output_image, f"{len(results)} faces with 5 keypoints", figsize=(12, 8))

#### 5점의 번호를 확대해 확인

점이 작아 어느 점이 몇 번인지 보이지 않는다. 1위 얼굴만 잘라 확대하고 번호를 적으면 `left`·`right` 가 화면
기준인지 인물 기준인지 눈으로 판정할 수 있다.

In [ ]:
x1, y1, x2, y2 = [int(value) for value in results[0].box]
margin = 30
crop = output_image[max(0, y1 - margin):y2 + margin, max(0, x1 - margin):x2 + margin]
zoomed = cv2.resize(crop, None, fx=3, fy=3, interpolation=cv2.INTER_LINEAR)

offset_x, offset_y = max(0, x1 - margin), max(0, y1 - margin)
for index, point in enumerate(results[0].keypoints):
    position = (int((point.x - offset_x) * 3) + 8, int((point.y - offset_y) * 3) - 8)
    cv2.putText(zoomed, str(index), position, cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 2,
                cv2.LINE_AA)

print("index 0 이 화면 왼쪽에 있으면 화면 기준, 오른쪽에 있으면 인물 기준이다.")
notebook_show_bgr(zoomed, "Top face - keypoint index (zoom x3)", figsize=(7, 8))